# Kinematics

运动学（Kinematics）是机器人操纵的几何基础，是连接高层任务规划与低层物理执行的关键桥梁。它描述了机器人的运动，而不考虑引起这些运动的力。本章将从运动学的基础概念出发，系统地探讨其理论、求解方法、优化策略，并最终揭示它在现代机器人核心挑战——任务与运动规划（TAMP）框架下的关键作用。

## 什么是运动学？

运动学是研究运动的几何学，它描述物体的位置、速度和加速度，而不考虑导致这些运动的力或质量。在机器人学中，运动学专注于机械臂的连杆和关节的几何关系。它包含了以下几个重要的概念：**构型空间** (Configuration Space, C-Space)，**正向运动学** (Forward Kinematics, FK)，**逆运动学** (Inverse Kinematics, IK)

- **构型空间** (Configuration Space, C-Space)：
构型空间（C-Space）是一个数学概念，代表了机器人所有可能状态的集合。对于一个由 `n` 个独立关节组成的机械臂，其构型可以用一个 `n` 维向量 $q = [q_1, q_2, ..., q_n]^T$ 来描述，这个向量中的每一个元素代表一个关节的位置。这个 `n` 维向量空间就是该机械臂的构型空间。从可视化的角度来看，C-Space 的一个重要优势在于它能够将复杂的机器人系统抽象成一个点，从而将后续更具挑战性的问题（如运动规划）转化为在构型空间中的路径搜索问题，使得求解过程更加直观和高效。

```{=html}
<iframe src="../web_demos/kinematics/c_space.html" title="configuration space 可视化"
        style="width: 100%; max-width: 1000px; height: 520px; border: 1px solid #e5e7eb; border-radius: 12px; box-shadow: 0 10px 25px rgba(0,0,0,0.08);
                display: block; margin: 0 auto; background: white;">
</iframe>
```



- **正向运动学 (Forward Kinematics, FK)**：
正向运动学 (FK) 是指给定一组关节构型（关节角度或位移）$q \in \mathbb{R}^n$（其中 $n$ 是机器人自由度），计算机器人末端执行器（End-Effector）在笛卡尔空间中的位姿（位置和姿态）$T \in SE(3)$ 的过程。这个映射函数是唯一且明确的：$T = f(q)$。

```{=html}
<iframe src="../web_demos/kinematics/fk/index.html" title="Forward Kinematics demo"
        style="width: 100%; max-width: 1200px; height: 520px; border: 1px solid #e5e7eb; border-radius: 12px; box-shadow: 0 10px 25px rgba(0,0,0,0.08);
                display: block; margin: 0 auto; background: white;">
</iframe>
```



-  **逆运动学 (Inverse Kinematics, IK)**：
逆运动学 (IK) 是 FK 的逆问题：给定末端执行器的期望位姿 $T$，求解所有能够达到该位姿的关节构型 $q$。即求解 $q = f^{-1}(T)$。与 FK 不同，IK 是一个极具挑战性的非线性问题，其核心挑战在于：
    - 解的存在性：并非所有期望位姿都是可达的（例如，超出机器人工作空间）。
    - 解的多样性：对于非冗余机器人，可能存在多个离散解（例如，"肘部向上"和"肘部向下"）。
    - 解的无穷性：对于冗余机器人（自由度 > 6），通常存在无限个解，形成一个连续的解空间。

```{=html}
<iframe src="../web_demos/kinematics/ik/index.html" title="Forward Kinematics demo"
        style="width: 100%; max-width: 1200px; height: 520px; border: 1px solid #e5e7eb; border-radius: 12px; box-shadow: 0 10px 25px rgba(0,0,0,0.08);
                display: block; margin: 0 auto; background: white;">
</iframe>
```

## 运动学的求解方法

### 正运动学

#### Denavit-Hartenberg (DH) 参数法：

DH 参数法是 20 世纪 50 年代提出的经典建模方法。它通过一套系统性的规则，在每个关节上建立一个坐标系。其核心思想是，从一个关节坐标系 $i-1$ 到下一个坐标系 $i$ 的变换 $T_{i-1}^i$ 可以**仅用 4 个参数**来描述：
- $a_{i-1}$：连杆长度 (link length)
- $\alpha_{i-1}$：连杆扭角 (link twist)
- $d_i$：连杆偏移 (link offset)
- $\theta_i$：关节转角 (joint angle)
总的正向运动学就是这些变换矩阵的连乘：
$$T_{base}^{EE} = T_0^1(q_1) \cdot T_1^2(q_2) \cdots T_{n-1}^n(q_n)$$

<figure style="text-align: center; margin: 20px 0;">
    <img src="https://daoming-chen.github.io/MP_lecture_assets/lec1/ur5e_dh.png" 
         width="70%" 
         style="display: block; margin: 0 auto;"> 
    <figcaption style="font-size: 16px; color: #555; margin-top: 10px; font-weight:">
        UR5e DH table <br>
        Kebria P M, Al-Wais S, Abdi H, et al. Kinematic and dynamic modelling of UR5 manipulator[C]//2016 IEEE international conference on systems, man, and cybernetics (SMC). IEEE, 2016: 004229-004234.
    </figcaption>
</figure>

#### **基于刚体变换链** (URDF)：

基于 URDF 的方法放弃了 DH 的 4 参数最小化约束，转而追求**建模的直观性和灵活性**。它的核心思想是：为机器人的每一个"连杆"(Link) 都定义一个固定的、符合直觉的坐标系，"关节"(Joint) 被定义为连接"父连杆"和"子连杆"的运动。从父连杆 $i-1$ 到子连杆 $i$ 的变换 $T_{i-1}^i(q_i)$ 由**静态（固定）变换**和**动态（可变）变换**组成。因此，总的正向运动学是一个由固定变换（连杆的几何结构）和可变变换（关节运动）交错组成的 $SE(3)$ 矩阵链。这种方法已经是当前机器人学中的实事标准，ROS, mujoco, issac 等主流平台均采用了这种直观的方式进行正运动学计算
$$T_{base}^{EE} = T_{base}^{L_1} \cdot T_{L_1}^{J_1} \cdot T_{J_1}(q_1) \cdot T_{J_1}^{L_2} \cdot T_{L_2}^{J_2} \cdot T_{J_2}(q_2) \cdots T_{J_{n-1}}^{L_n} \cdot T_{L_n}^{EE}$$

<figure style="text-align: center; margin: 20px 0;">
    <img src="https://daoming-chen.github.io/MP_lecture_assets/lec1/tf2_broadcaster_with_stow.gif" 
         width="70%" 
         style="display: block; margin: 0 auto;"> 
    <figcaption style="font-size: 16px; color: #555; margin-top: 10px; font-weight:">
        <a href="https://docs.hello-robot.com/0.2/stretch-tutorials/ros1/example_10/" target="_blank">hello-robot stretch-tutorials tf2 example</a>
    </figcaption>
</figure>


#### **指数积** (Product of Exponentials, PoE)：

PoE 方法，也常与**螺旋理论 (Screw Theory)** 相关联，是现代机器人学教科书中推崇的方法。它提供了一个优美且不依赖于中间坐标系的表述方式。核心思想是：定义一个 **零位形** $M$（即当所有 $q_i = 0$ 时的末端执行器位姿）和**关节"螺旋" (Twist)** $\mathcal{V}_i \in se(3)$（每一个关节的运动可以被描述为一个 6 维的螺旋向量）。任意关节构型 $q$ 下的位姿 $T(q)$ 可以通过指数映射计算：
$$T(q) = e^{[\mathcal{V}_1]q_1} \cdot e^{[\mathcal{V}_2]q_2} \cdots e^{[\mathcal{V}_n]q_n} \cdot M$$
其中 $e^{[\mathcal{V}_i]q_i}$ 是一个 $SE(3)$ 矩阵，表示绕着螺旋轴 $\mathcal{V}_i$ 运动 $q_i$ 的量。

#### 方法对比总结

| 方法 | 核心思想 | 优点 | 缺点 | 主要应用 |
| :--- | :--- | :--- | :--- | :--- |
| **DH 参数法** | 4 参数最小化坐标系变换 | 参数少，形式统一 | 建模不直观，有奇异性 | 传统机器人学分析 |
| **URDF 变换链** | 直观的刚体变换连乘 | **直观，灵活，生态兼容** | 参数冗余 | **工程实现，仿真，ROS** |
| **指数积 (PoE)** | 关节螺旋与零位形 | **理论优雅，几何直观** | 学习门槛高 | 现代机器人学理论，分析推导 |


### 逆运动学